# YSH-007 시각 검토

기존 CSV 산출물을 읽어 분포와 군집을 직접 검토하는 notebook입니다. GMM, scaler, 잔차 모델을 학습하거나 설정을 다시 선택하지 않습니다. 날짜별 Raw4 분포와 원 좌표 산점도에만 원본 Excel의 네 변수를 연결합니다. 날짜는 그림의 색·분할에만 쓰며 GMM 입력 변수는 아닙니다.

프로젝트의 공용 .venv 커널을 선택하고 위에서 아래로 실행하세요. 00의 필터 값을 바꾼 뒤 보고 싶은 셀만 다시 실행할 수 있습니다. Plotly 그림에서 확대·범례 클릭·행 hover가 가능합니다.

주의: 전체 자료 재적합 군집은 in-sample 서술이고, 날짜 제외 점수는 교차 적합 서술입니다. 두 결과를 독립 검증이나 행별 불량 정답으로 읽지 않습니다. outputs 폴더는 Git에서 제외되므로 나중에 같은 결과를 보려면 CSV 산출물을 보존해야 합니다.

## 00. Load outputs / 검토 범위

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

def find_experiment():
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        for candidate in (base, base / "experiments" / "YSH" / "ysh-007"):
            if (candidate / "config.json").is_file() and (candidate / "outputs").is_dir():
                return candidate
    raise FileNotFoundError("YSH-007 폴더를 찾지 못했습니다. F:/Kamp 또는 실험 폴더에서 notebook을 여세요.")

EXP = find_experiment()
ROOT = EXP.parents[2]
OUT = EXP / "outputs"
CONFIG = json.loads((EXP / "config.json").read_text(encoding="utf-8-sig"))
MANIFEST = json.loads((OUT / "manifest.json").read_text(encoding="utf-8"))
VERIFICATION = json.loads((OUT / "verification.json").read_text(encoding="utf-8"))
for name, expected in VERIFICATION["key_output_sha256"].items():
    with (OUT / name).open("rb") as source:
        actual = hashlib.file_digest(source, "sha256").hexdigest()
    if actual != expected:
        raise ValueError(f"?? ??? ??? ????: {name}")
REQUIRED = [
    "in_sample_scores.csv", "lodo_row_scores.csv", "daily_raw4.csv",
    "daily_component_share.csv", "selected_component_profiles.csv",
    "lodo_pattern_rows.csv", "residual_pattern_models.csv",
    "daily_pattern_share.csv", "regime_confounding.csv",
    "result_association_main.csv", "row_cluster_summary.csv",
]
missing = [name for name in REQUIRED if not (OUT / name).is_file()]
if missing:
    raise FileNotFoundError(f"기존 산출물이 없습니다: {missing}. 실험 결과를 먼저 준비하세요.")
TABLE = {name.removesuffix(".csv"): pd.read_csv(OUT / name) for name in REQUIRED}
if MANIFEST.get("status") != "verified":
    print("주의: manifest 상태가 verified가 아닙니다:", MANIFEST.get("status"))
print("실험 폴더:", EXP)
print("커널:", sys.executable)
print("선택 설정:", MANIFEST["selected_config"])
print("읽은 CSV:", len(TABLE), "개")

# 빈 목록은 전체를 뜻합니다. 값을 바꾸고 필요한 그림 셀만 다시 실행하세요.
FOCUS_DATES = []                 # 예: ["2020-03-25", "2020-04-03"]
FOCUS_REGIMES = []               # 예: ["B"]
FOCUS_COMPONENTS = []            # 예: [0, 3]
SCATTER_VIEW = "all"             # "all" 또는 "unique4"
NLL_SCATTER_COLOR = "tail"       # "tail", "regime", "date", "component"
PATTERN_POLICY = "global"        # "global" 또는 "component"

def focus(frame, component_col=None):
    view = frame
    if FOCUS_DATES and "date" in view:
        view = view[view["date"].isin(FOCUS_DATES)]
    if FOCUS_REGIMES and "regime" in view:
        view = view[view["regime"].isin(FOCUS_REGIMES)]
    if FOCUS_COMPONENTS and component_col and component_col in view:
        view = view[view[component_col].isin(FOCUS_COMPONENTS)]
    return view.copy()

def show_table(frame):
    display(frame.reset_index(drop=True))

## 01. 날짜별 Raw4 분포

Raw4 원본 좌표가 필요한 유일한 입력은 원본 Excel입니다. 해시가 실험 당시와 다르면 멈춥니다. 날짜별 violin은 각 날짜의 분포를, A/B histogram은 행 수로 가중된 전체 형태를 보여줍니다. 정확 반복이 많으므로 histogram의 높이를 독립 표본 수처럼 읽지 않습니다.

In [ ]:
RAW_PATH = ROOT / "data" / "Welding_Data_Set_01.xlsx"
if not RAW_PATH.is_file():
    raise FileNotFoundError(f"원본 Excel이 없습니다: {RAW_PATH}")
with RAW_PATH.open("rb") as source:
    digest = hashlib.file_digest(source, "sha256").hexdigest()
if digest != CONFIG["raw_sha256"]:
    raise ValueError("Excel SHA-256이 기존 YSH-007 산출물의 입력과 다릅니다.")
raw_excel = pd.read_excel(RAW_PATH, sheet_name="Raw data")
raw_excel["excel_row"] = np.arange(len(raw_excel)) + 2
raw_excel["date"] = pd.to_datetime(raw_excel["working time"]).dt.strftime("%Y-%m-%d")
regime_map = {date: regime for regime, dates in CONFIG["regime"].items() for date in dates}
raw_excel["regime"] = raw_excel["date"].map(regime_map)
short = ["F", "I", "V", "t"]
raw4 = raw_excel[["excel_row", "date", "regime", *CONFIG["features"]]].rename(
    columns=dict(zip(CONFIG["features"], short)))
raw4["repeat_count"] = raw4.groupby(short)["excel_row"].transform("size")
assert len(raw4) == 11939 and raw4["excel_row"].is_unique
in_sample = raw4.merge(
    TABLE["in_sample_scores"].drop(columns=["date", "regime"]),
    on="excel_row", validate="one_to_one")
lodo = raw4.merge(
    TABLE["lodo_row_scores"].drop(columns=["date", "regime"]),
    on="excel_row", validate="one_to_one")
assert len(in_sample) == len(lodo) == len(raw4)
print(f"Raw {len(raw4):,}행; 고유 Raw4 {raw4[short].drop_duplicates().shape[0]:,}개")
show_table(TABLE["daily_raw4"][["date", "regime", "raw_rows", "unique4_rows", "high_force_rows"]])

In [ ]:
raw_view = focus(raw4)
if raw_view.empty:
    raise ValueError("필터 조건에 해당하는 Raw 행이 없습니다.")
for feature in short:
    fig = px.violin(
        raw_view, x="date", y=feature, color="regime", box=True, points=False,
        category_orders={"date": CONFIG["dates"], "regime": ["A", "B"]},
        title=f"{feature}: 날짜별 Raw 분포")
    fig.update_layout(height=420)
    fig.show()
for feature in ["I", "V"]:
    fig = px.histogram(
        raw_view, x=feature, color="regime", barmode="overlay",
        histnorm="probability density", nbins=65, opacity=0.55,
        title=f"{feature}: A/B 전체 행 분포 (행 가중)")
    fig.show()

## 02. GMM component가 원 좌표에서 차지하는 위치

아래 세 산점도는 날짜와 무관하게 전체 11,939행에 재적합한 GMM의 hard assignment입니다. 범례를 눌러 component를 끄거나 켜세요. 같은 Raw4 값은 정확히 포개집니다. SCATTER_VIEW를 unique4로 바꾸면 고유 조합 한 점씩 볼 수 있지만 그때는 행 빈도가 사라집니다. K=6이 실제 공정 여섯 개라는 뜻은 아닙니다.

In [ ]:
comp_view = focus(in_sample, "assigned_component")
if SCATTER_VIEW == "unique4":
    comp_view = comp_view.drop_duplicates(short)
elif SCATTER_VIEW != "all":
    raise ValueError("SCATTER_VIEW는 all 또는 unique4여야 합니다.")
comp_view["component"] = comp_view["assigned_component"].map(lambda x: f"C{x}")
colors = {f"C{k}": color for k, color in enumerate(px.colors.qualitative.Safe[:6])}
for xvar, yvar in [("F", "I"), ("F", "V"), ("I", "V")]:
    fig = px.scatter(
        comp_view, x=xvar, y=yvar, color="component",
        color_discrete_map=colors, render_mode="webgl", opacity=0.38,
        hover_data=["excel_row", "date", "regime", "repeat_count", "posterior_max"],
        title=f"Raw4 전체 재적합 GMM: {xvar}–{yvar} ({len(comp_view):,}개 표시)")
    fig.update_traces(marker={"size": 5})
    fig.update_layout(height=520)
    fig.show()
show_table(TABLE["row_cluster_summary"])

### A/B별 posterior 확신도

날짜 제외 모델의 최대 posterior와 1−최대 posterior를 확인합니다. 값이 1에 몰리더라도 정확 복제와 수치적으로 분리된 성분의 영향일 수 있습니다. component 번호는 원 단위 평균을 이용해 날짜별 모델 사이에서 대응시킨 것이므로 완벽한 물리 라벨은 아닙니다.

In [ ]:
posterior = focus(lodo, "assigned_component")
show_table(posterior.groupby("regime", as_index=False).agg(
    rows=("excel_row", "size"),
    mean_max_posterior=("posterior_max", "mean"),
    share_below_090=("posterior_max", lambda s: (s < 0.90).mean()),
    share_below_099=("posterior_max", lambda s: (s < 0.99).mean())))
fig = px.violin(
    posterior, x="date", y="posterior_max", color="regime", box=True,
    points=False, category_orders={"date": CONFIG["dates"]},
    title="LODO: 날짜별 최대 posterior")
fig.show()
posterior["log10_ambiguity"] = np.log10(np.clip(1 - posterior["posterior_max"], 1e-12, None))
fig = px.histogram(
    posterior, x="log10_ambiguity", color="regime", barmode="overlay",
    histnorm="probability density", nbins=70, opacity=0.55,
    title="LODO: log10(1 − 최대 posterior), 왼쪽일수록 확신도가 높음")
fig.show()
share = TABLE["daily_component_share"]
share = share[share["phase"].eq("lodo")].set_index("date").reindex(CONFIG["dates"])
fig = px.imshow(
    share[[f"C{k}_posterior_share" for k in range(6)]],
    color_continuous_scale="Viridis", aspect="auto",
    labels={"x": "component", "y": "date", "color": "평균 posterior"},
    title="LODO 날짜×component 평균 posterior")
fig.show()

## 03. NLL과 Mahalanobis d²가 가리키는 행

전역 NLL은 혼합분포 전체에서 드문 정도, d²는 배정 component 안에서 먼 정도입니다. 색상 범례의 NLL only / d² only를 끄고 켜며 원본 F/I/V/t와 날짜를 hover로 확인하세요. 상위 10% 경계는 전체 LODO 행을 합친 **기술적 기준**입니다. 날짜마다 fit 모델이 달라 아래 오른쪽 train 분위 산점도도 함께 확인합니다.

In [ ]:
score = focus(lodo, "assigned_component")
all_nll_q90 = lodo["global_nll"].quantile(0.90)
all_d2_q90 = lodo["mahalanobis_d2"].quantile(0.90)
n_high = score["global_nll"].ge(all_nll_q90)
d_high = score["mahalanobis_d2"].ge(all_d2_q90)
score["tail"] = np.select(
    [n_high & d_high, n_high & ~d_high, ~n_high & d_high],
    ["both", "NLL only", "d2 only"], default="neither")
score["component"] = score["assigned_component"].map(lambda x: f"C{x}")
color_col = {
    "tail": "tail", "regime": "regime", "date": "date",
    "component": "component",
}.get(NLL_SCATTER_COLOR)
if color_col is None:
    raise ValueError("NLL_SCATTER_COLOR 값을 확인하세요.")
show_table(pd.crosstab(score["regime"], score["tail"]))
show_table(pd.crosstab(score["component"], score["tail"]))
show_table(score.groupby("tail", as_index=False).agg(
    rows=("excel_row", "size"),
    high_F_share=("F", lambda s: (s > 3).mean()),
    mean_F=("F", "mean"),
    mean_I=("I", "mean"),
    mean_V=("V", "mean"),
    mean_t=("t", "mean")))
fig = px.scatter(
    score, x="global_nll", y="mahalanobis_d2", color=color_col,
    render_mode="webgl", opacity=0.40,
    hover_data=["excel_row", "date", "regime", "component", "F", "I", "V", "t"],
    title="LODO 행별 NLL vs 배정 component d²")
fig.update_traces(marker={"size": 5})
fig.add_vline(x=all_nll_q90, line_dash="dash", line_color="black")
fig.add_hline(y=all_d2_q90, line_dash="dash", line_color="black")
fig.show()
fig = px.scatter(
    score, x="global_nll_train_percentile",
    y="mahalanobis_d2_train_percentile", color="regime",
    render_mode="webgl", opacity=0.35,
    hover_data=["excel_row", "date", "component", "F", "I", "V", "t"],
    title="각 날짜의 train 분포 기준 분위: NLL vs d²")
fig.update_traces(marker={"size": 5})
fig.show()

### Component score audit

Full refit and leave-one-day-out scoring are shown separately. The top-10% cutoffs are pooled across all rows within each phase. A small or low-density component may have high global NLL throughout, which is not a within-component error rate. LODO component IDs were aligned by their raw means and can shift across folds.


In [ ]:
def cluster_score_audit(frame, phase):
    nll_cut = frame["global_nll"].quantile(0.90)
    d2_cut = frame["mahalanobis_d2"].quantile(0.90)
    audit = frame.assign(
        nll_top=frame["global_nll"].ge(nll_cut),
        d2_top=frame["mahalanobis_d2"].ge(d2_cut)
    ).groupby("assigned_component", as_index=False).agg(
        rows=("excel_row", "size"),
        nll_p95=("global_nll", lambda s: s.quantile(0.95)),
        d2_p95=("mahalanobis_d2", lambda s: s.quantile(0.95)),
        nll_top_rows=("nll_top", "sum"),
        nll_top_rate=("nll_top", "mean"),
        d2_top_rows=("d2_top", "sum"),
        d2_top_rate=("d2_top", "mean"))
    audit["component"] = audit["assigned_component"].map(lambda x: f"C{x}")
    print(f"{phase}: global NLL q90={nll_cut:.3f}, d2 q90={d2_cut:.3f}")
    show_table(audit.round(3))
    chart = audit.melt(
        id_vars="component", value_vars=["nll_top_rate", "d2_top_rate"],
        var_name="score", value_name="share")
    px.bar(chart, x="component", y="share", color="score", barmode="group",
           title=f"{phase}: within-cluster share exceeding pooled q90").show()
    return audit

audit_full = cluster_score_audit(TABLE["in_sample_scores"], "Full refit (in-sample)")
audit_lodo = cluster_score_audit(TABLE["lodo_row_scores"], "LODO")


## 04. 날짜별 점수 분포 전체

p95 표의 한 숫자 뒤에 전체 이동이 있는지, 일부 꼬리만 긴지 확인합니다. 각 날짜에 다른 8일 모델이 적용됐고 날짜 간 정확 같은 네 값이 많습니다. 아래 그림은 외부 신규 생산일 성능이 아닙니다.

In [ ]:
daily_score_view = focus(lodo, "assigned_component")
for metric, title in [
    ("global_nll", "LODO 전역 NLL"),
    ("mahalanobis_d2", "LODO component 내부 d²"),
]:
    fig = px.violin(
        daily_score_view, x="date", y=metric, color="regime",
        box=True, points=False,
        category_orders={"date": CONFIG["dates"]},
        title=f"{title}: 날짜별 전체 분포")
    fig.update_layout(height=480)
    fig.show()
    fig = px.ecdf(
        daily_score_view, x=metric, color="date",
        category_orders={"date": CONFIG["dates"]},
        title=f"{title}: 날짜별 ECDF")
    fig.show()

## 05. Residual pattern 분리 여부

먼저 전수 재적합 기준 주 후보 풀의 평균 잔차 방향을 봅니다. heatmap 색은 변수별 Raw 표준편차로 나눈 값이고 hover에는 실제 원 단위 잔차가 있습니다. 이어서 날짜 제외 후보 행의 원 단위 잔차를 직접 산점도로 봅니다. 날짜별 pattern label 정렬이 불안정할 수 있으므로 색이 겹치면 분리 주장을 낮춰야 합니다.

In [ ]:
reference = TABLE["residual_pattern_models"].query(
    "phase == 'reference_in_sample' and policy == 'global'").sort_values("pattern")
residual_cols = [f"residual_{f}" for f in short]
raw_values = reference[residual_cols].to_numpy(float)
z = raw_values / raw4[short].std().to_numpy(float)
fig = go.Figure(go.Heatmap(
    z=z, x=short, y=[f"P{k}" for k in reference["pattern"]],
    customdata=raw_values,
    colorscale="RdBu", zmid=0,
    hovertemplate="pattern=%{y}<br>feature=%{x}<br>원 단위 잔차=%{customdata:.4g}<br>Raw SD 단위=%{z:.3g}<extra></extra>"))
fig.update_layout(title="전수 재적합 잔차 profile (표시용 표준화)", height=420)
fig.show()

if PATTERN_POLICY not in {"global", "component"}:
    raise ValueError("PATTERN_POLICY는 global 또는 component여야 합니다.")
candidates = TABLE["lodo_pattern_rows"]
candidates = candidates[candidates["policy"].eq(PATTERN_POLICY)]
candidates = candidates.merge(
    lodo[["excel_row", "date", "assigned_component"]],
    on=["excel_row", "date"], validate="one_to_one")
candidates = candidates.merge(
    raw4[["excel_row", *short]], on="excel_row", validate="one_to_one")
profiles = TABLE["selected_component_profiles"]
profiles = profiles[profiles["phase"].eq("lodo")][
    ["date", "component", *[f"mean_{f}" for f in short]]]
candidates = candidates.merge(
    profiles, left_on=["date", "assigned_component"],
    right_on=["date", "component"], validate="many_to_one")
assert not candidates[[f"mean_{f}" for f in short]].isna().any().any()
for feature in short:
    candidates[f"residual_{feature}"] = (
        candidates[feature] - candidates[f"mean_{feature}"])
candidates = focus(candidates, "assigned_component")
candidates["pattern_name"] = candidates["pattern"].map(lambda x: f"P{x}")
print(f"날짜 제외 {PATTERN_POLICY} 후보 행: {len(candidates):,}")
for xvar, yvar in [("residual_F", "residual_t"), ("residual_I", "residual_V")]:
    fig = px.scatter(
        candidates, x=xvar, y=yvar, color="pattern_name",
        facet_col="regime", render_mode="webgl", opacity=0.55,
        hover_data=["excel_row", "date", "assigned_component", "F", "I", "V", "t"],
        title=f"LODO 잔차 후보: {xvar} vs {yvar}")
    fig.update_traces(marker={"size": 6})
    fig.show()

In [ ]:
shares = TABLE["daily_pattern_share"]
cols = [f"pattern_{k}_share_all" for k in range(5)]
share_long = shares.melt(
    id_vars=["date", "regime", "policy"], value_vars=cols,
    var_name="pattern", value_name="share_all")
share_long = focus(share_long)
fig = px.line(
    share_long, x="date", y="share_all", color="pattern",
    facet_col="policy", markers=True,
    category_orders={"date": CONFIG["dates"]},
    title="전역 상위 10% vs component별 상위 10%: 날짜별 pattern 비중")
fig.show()

## 06. Result 일별 overlay

Result는 날짜별 기재 건수입니다. 03-27은 전 유형이 미기록, 03-31은 유형3 미기록, 04-07 유형1은 실제 기록된 0입니다. 각 점은 날짜 하나이므로 이 그림으로 행별 불량을 판단할 수 없습니다. A/B 색의 분리를 먼저 살펴보세요.

In [ ]:
quality = focus(TABLE["regime_confounding"])
quality_long = quality.melt(
    id_vars=["date", "regime", "global_nll_p95", "mahalanobis_d2_p95", "raw_rows"],
    value_vars=["type1", "type2", "type3"],
    var_name="defect_type", value_name="recorded_count").dropna(
        subset=["recorded_count"])
for metric in ["global_nll_p95", "mahalanobis_d2_p95"]:
    fig = px.scatter(
        quality_long, x=metric, y="recorded_count", color="regime",
        facet_col="defect_type", hover_data=["date", "raw_rows"],
        title=f"날짜별 {metric} vs Result 기재 건수")
    fig.update_traces(marker={"size": 12})
    fig.show()
associations = TABLE["result_association_main"]
associations = associations[
    associations["target"].eq("count") &
    associations["metric"].isin(["global_nll_p95", "mahalanobis_d2_p95"])]
show_table(associations[[
    "scope", "defect_type", "metric", "n_dates", "rho", "loo_min", "loo_max"]])

## ? ?? ?? ? Codex

- ??? violin?? B? F? ?? ? ??? ??? ????. I? V? A?? ?? ??? ???, ?? V? ???? ?? ???? ????? ??? ??. t? ?? ?? ?? 71/72ms? ?????.
- ? ?? ????? C0?C3? ?? F ??? ????. ?? F ??? C1?C2?C4?C5? ?? 2?? ???? ????? ???. ????? ?? ?? ?? ?? ??? ??? ? ??.
- LODO ?? posterior ???? A 0.9998, B 0.9993??. 0.90 ??? A 2.2%, B 4.6%? ???? hard assignment? ????? ?????, ?? ?? ?? ??? ??? ???.
- ?? ?? ?? 10% ?? NLL only 992? ? 968?? B?? F>3bar ??? 76.3%?. d? only 999??? A 436??B 563?? ??? F>3bar ??? 0.5%?. NLL? d?? ??? ?? ?? ??? ?? ????.
- ?? ?? ?? ??? F?t ? I?V ???? P1?P3? ??? ?? ?? ???. ? profile? ???? ?? ????? ??? ? ???? ??? ???. ?? ? ??????? ?? ???? ??.

? ??? ?? ??? ?? ?? ???. ?? ???? ??? ??? ? ??? Excel ? ??? ?? ???.


## 07. 내가 직접 적는 관찰 메모

그림을 확대하고 범례에서 군집을 하나씩 켜 보며 아래 항목을 채우세요. 관찰과 해석을 구분하고, 날짜·component·Excel 행 번호를 남기면 다음 실험에서 그대로 찾을 수 있습니다.

- B의 차이는 F에서만 선명한가? I/V에서도 같은 방향의 분포 이동이 보이는가? **관찰:**
- K=6 성분 중 한 연속 분포를 잘게 나눈 것으로 보이는 쌍은? **관찰:**
- posterior가 낮아 hard cluster가 애매한 행·날짜는? **관찰:**
- NLL only 행은 어떤 날짜·A/B·component·Raw 범위에 있는가? **관찰:**
- d² only 행은 어떤 날짜·A/B·component·Raw 범위에 있는가? **관찰:**
- B의 점수는 전체 분포가 이동했는가, 꼬리만 달라졌는가? **관찰:**
- P1·P3처럼 t가 양의 잔차인 profile은 점 구름에서도 겹치는가? **관찰:**
- Result overlay에서 A/B를 따로 보면 관계가 남는가? **관찰:**

추후 다시 볼 때는 00의 필터만 바꾸고 원하는 셀을 재실행하면 됩니다. CSV와 원본 Excel이 유지되는 한 GMM을 다시 학습할 필요가 없습니다.

## 08. Physical proxy and ordered-neighbor follow-up

YSH-007의 사후 검증 산출물을 모델 재학습 없이 읽는다. 행 후보는 공정 오류 라벨이 아니며 3/4/8/16행 창은 실제 제품 사이클로 확정되지 않았다. 아래 그림에서 날짜와 원본 Excel 행 번호를 직접 바꿔 확인한다.

In [ ]:

from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import display

FOLLOWUP_DIR = Path.cwd()
if not (FOLLOWUP_DIR / "outputs" / "physical_row_followup.csv").exists():
    FOLLOWUP_DIR = Path("experiments/YSH/ysh-007").resolve()
FU_OUT = FOLLOWUP_DIR / "outputs"
FU_ROWS = pd.read_csv(FU_OUT / "physical_row_followup.csv")
FU_DAILY = pd.read_csv(FU_OUT / "physical_daily_result.csv")
FU_LAG = pd.read_csv(FU_OUT / "physical_sequence_sensitivity.csv")
FU_NEIGHBOR = pd.read_csv(FU_OUT / "physical_neighbor_summary.csv")
FU_CYCLE = pd.read_csv(FU_OUT / "physical_cycle_local_windows.csv")
assert len(FU_ROWS) == 11939 and FU_ROWS["excel_row"].is_unique
sig_cols = ["indentation_rows", "insufficient_rows", "crack_related_rows"]
sig_view = FU_DAILY[["date", "regime", "rows", *sig_cols]].melt(
    id_vars=["date", "regime", "rows"], var_name="signature", value_name="candidate_rows")
sig_view["candidate_share"] = sig_view["candidate_rows"] / sig_view["rows"]
fig = px.bar(sig_view, x="date", y="candidate_share", color="signature",
             barmode="group", hover_data=["regime", "candidate_rows", "rows"],
             title="Result-blind physical signatures by date")
fig.show()
display(FU_DAILY[["date", "regime", "review_rows", *sig_cols,
                  "type1", "type2", "type3"]])


In [ ]:

lag_view = FU_LAG.groupby("lag", as_index=False)[
    ["same", "pairs", "unique_endpoint_same", "unique_endpoint_pairs"]].sum()
lag_view["all_pairs"] = lag_view["same"] / lag_view["pairs"]
lag_view["unique_endpoints"] = (
    lag_view["unique_endpoint_same"] / lag_view["unique_endpoint_pairs"])
fig = px.line(lag_view.melt(id_vars="lag", value_vars=["all_pairs", "unique_endpoints"],
                            var_name="pair_set", value_name="same_component_rate"),
              x="lag", y="same_component_rate", color="pair_set", markers=True,
              title="GMM component recurrence; lag is not a cycle ID")
fig.show()
display(FU_CYCLE[["date", "start_excel_row", "end_excel_row",
                  "lag3_peak", "phase3_adjusted_mi"]]
        .sort_values("lag3_peak", ascending=False).head(12))


In [ ]:

ANCHOR_EXCEL_ROW = 4830  # Change to any Excel row from FU_ROWS
CONTEXT_WIDTH = 3        # 3, 4, 8, or 16
anchor = FU_ROWS.loc[FU_ROWS.excel_row.eq(ANCHOR_EXCEL_ROW)].iloc[0]
context = FU_ROWS[
    FU_ROWS.safe_segment.eq(anchor.safe_segment)
    & FU_ROWS.excel_row.between(ANCHOR_EXCEL_ROW - 2*CONTEXT_WIDTH,
                               ANCHOR_EXCEL_ROW + 2*CONTEXT_WIDTH)
].copy()
long = context.melt(
    id_vars=["excel_row", "date", "regime", "assigned_component",
             "row_review_candidate"],
    value_vars=["F", "I", "V", "t"], var_name="column", value_name="value")
fig = px.line(long, x="excel_row", y="value", facet_row="column",
              height=850, title=f"Raw4 around Excel row {ANCHOR_EXCEL_ROW}")
fig.add_vline(x=ANCHOR_EXCEL_ROW, line_dash="dash", line_color="red")
fig.update_yaxes(matches=None)
fig.show()
display(context[["excel_row", "date", "assigned_component", "F", "I", "V", "t",
                 "E_proxy_j", "global_nll_train_percentile",
                 "mahalanobis_d2_train_percentile", "conditional_z_F", "conditional_z_I", "conditional_z_V", "conditional_z_t", "max_abs_conditional_z",
                 "row_review_candidate"]])
display(FU_NEIGHBOR[(FU_NEIGHBOR.anchor_excel_row == ANCHOR_EXCEL_ROW)
                    & (FU_NEIGHBOR.width == CONTEXT_WIDTH)])


### 08.1 직접 검토 메모

- 선택한 행은 군집 내부 d²가 큰가, 전역 NLL만 큰가?
- 같은 물리 signature와 큰 점수가 이전·다음 창에도 이어지는가? 창 위상/폭을 바꾸면 달라지는가?
- 같은 Raw4 조합 또는 같은 주변 배열이 다른 날짜에도 나타나는가?
- 3행 성분 재등장이 국소 위상과 원본 변수에서도 보이는가?
- Result가 기록된 날짜인가? A/B 안에서도 방향이 유지되는가?

관찰할 Excel 행 번호, 반례, 필요한 설비/검사 로그를 여기에 직접 적는다.